# Music Generation with AI - Demo Notebook

This notebook demonstrates the music generation system using LSTM neural networks.

## Steps:
1. Load MIDI files
2. Preprocess into sequences
3. Build LSTM model
4. Train the model
5. Generate new music

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

from src.data_preprocessor import MidiPreprocessor
from src.model import MusicLSTM
from src.generate import MusicGenerator
from src.utils import ensure_directories

## 1. Setup and Load Data

In [ ]:
# Ensure directories exist
ensure_directories()

# Load MIDI files
data_dir = '../data/midi'
preprocessor = MidiPreprocessor()
preprocessor.load_midi_files(data_dir)

print(f"\nTotal notes extracted: {len(preprocessor.notes)}")
print(f"Vocabulary size: {len(preprocessor.vocabulary)}")
print(f"\nFirst 20 notes: {preprocessor.notes[:20]}")

## 2. Prepare Training Sequences

In [ ]:
# Prepare sequences
sequence_length = 50
X, y = preprocessor.prepare_sequences(sequence_length)

print(f"\nInput shape (X): {X.shape}")
print(f"Output shape (y): {y.shape}")
print(f"Number of training samples: {len(X)}")

## 3. Build LSTM Model

In [ ]:
# Create and build model
vocabulary_size = len(preprocessor.vocabulary)
model = MusicLSTM(vocabulary_size, sequence_length)
model.build_simple_model(lstm_units=128)

print("\nModel built successfully!")

## 4. Train the Model

In [ ]:
# Train model (quick demo with 10 epochs)
print("Starting training...")
history = model.train(X, y, epochs=10, batch_size=32, validation_split=0.2)

print(f"\nTraining complete!")
print(f"Final accuracy: {history['accuracy'][-1]:.2%}")

## 5. Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(history['loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(history['accuracy'], label='Train')
axes[1].plot(history['val_accuracy'], label='Validation')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Generate Music

In [ ]:
# Create generator
generator = MusicGenerator(model, preprocessor.vocabulary, preprocessor.inverse_vocabulary)

# Generate music
output_path = '../generated/demo_generated.mid'
print(f"Generating music to: {output_path}")

generator.generate_and_save(
    output_path=output_path,
    length=200,
    temperature=0.8,
    instrument_type='piano'
)

## 7. What's Next?

To improve the model:

1. **More data**: Add more MIDI files to `data/midi/`
2. **Longer training**: Increase epochs to 50-100
3. **Larger model**: Use `--lstm_units 256` or more
4. **Different temperatures**: Try 0.5-1.0 for different results

### Command line usage:

```bash
# Train with full model
python train.py --epochs 50 --lstm_units 256

# Generate music
python generate_music.py --length 500 --temperature 0.7
```